<a href="https://colab.research.google.com/github/TaShapovalova/my-colab-project/blob/main/MODUL_2_case4_%D0%B2%D0%BE%D0%BF%D1%80%D0%BE%D1%81%D1%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://githubusercontent.com

--2026-06-07 14:43:39--  https://githubusercontent.com/
Resolving githubusercontent.com (githubusercontent.com)... failed: No address associated with hostname.
wget: unable to resolve host address ‘githubusercontent.com’


In [ ]:
import pandas as pd
from google.colab import files
from collections import Counter

print("Please upload your dataset file. For example, a CSV file.")
uploaded = files.upload()

df = None
for filename in uploaded.keys():
  try:
    # Assuming the uploaded file is a CSV. Adjust if it's a different format.
    # Using engine='python' and on_bad_lines='skip' for more robust CSV parsing
    # to handle potential formatting errors like 'Error tokenizing data'.
    df = pd.read_csv(filename, engine='python', on_bad_lines='skip')
    print(f"\nSuccessfully loaded '{filename}'. Here are the first 5 rows:")
    print(df.head())
    break # Assuming only one file needs to be processed for the initial request
  except Exception as e:
    print(f"Error reading '{filename}': {e}")

# The original code for processing groceries.csv is now commented out or removed
# as it is replaced by the file upload logic.
# If further processing of the uploaded 'df' is needed, it can be added here.

# Example of how to integrate with the original Counter logic if needed:
# if df is not None and not df.empty:
#     # This part depends on the structure of the uploaded CSV.
#     # For now, I will just print the head as requested.
#     pass

Please upload your dataset file. For example, a CSV file.


Saving groceries.csv to groceries.csv

Successfully loaded 'groceries.csv'. Here are the first 5 rows:
       citrus fruit semi-finished bread       margarine  \
0    tropical fruit              yogurt          coffee   
1        whole milk                None            None   
2         pip fruit              yogurt    cream cheese   
3  other vegetables          whole milk  condensed milk   
4        rolls/buns                None            None   

                ready soups  
0                      None  
1                      None  
2              meat spreads  
3  long life bakery product  
4                      None  


In [ ]:
import collections

# Flatten the DataFrame into a single list of all products, ignoring None values
all_products = []
for col in df.columns:
    # Using .dropna() to remove None/NaN values before extending the list
    # Convert all values to string to handle mixed types gracefully
    all_products.extend(df[col].dropna().astype(str).tolist())

# Count the frequency of each product
product_counts = collections.Counter(all_products)

# Get products sorted by frequency in descending order
most_common_products = product_counts.most_common()

# Check if there are at least 59 products
if len(most_common_products) >= 59:
    # Get the 59th most frequent product (index 58 because it's 0-indexed)
    fifty_ninth_most_frequent = most_common_products[58]
    print(f"The 59th most frequent product is: {fifty_ninth_most_frequent[0]} (with {fifty_ninth_most_frequent[1]} occurrences).")
else:
    print("There are fewer than 59 unique products in the dataset.")

The 59th most frequent product is: photo/film (with 60 occurrences).


In [ ]:
# 2. Находим 59-е место (индекс 58, так как нумерация с 0)
if len(most_common_products) >= 59:
    product_59_tuple = most_common_products[58]
    product_name = product_59_tuple[0]
    product_frequency = product_59_tuple[1]
    print(f"Продукт на 59 месте: {product_name}")
    print(f"Его частота: {product_frequency}")
else:
    print("В наборе данных менее 59 уникальных продуктов.")

Продукт на 59 месте: photo/film
Его частота: 60


In [ ]:
# 1. Объединяем все колонки в одну строку для каждой транзакции, удаляя NaN
transactions = df.apply(lambda row: ','.join(row.dropna().astype(str)), axis=1)

# 2. Разбиваем строки на отдельные товары
all_items = transactions.str.split(',').explode()

# 3. Убираем лишние пробелы вокруг названий товаров (важно!)
all_items = all_items.str.strip()

# 4. Удаляем пустые строки, если они остались
all_items = all_items[all_items != '']

# 5. Считаем частоту и смотрим 59-е место (индекс 58)
frequencies = all_items.value_counts()
print("59-е место в Python (по логике R):", frequencies.index[58])
print("Частота:", frequencies.iloc[58])

59-е место в Python (по логике R): photo/film
Частота: 60


In [ ]:
# Читаем файл построчно, чтобы не нарушать структуру транзакций
with open('groceries.csv', 'r') as f:
    lines = f.readlines()

# Разделяем товары в каждой строке по запятой и очищаем от пробелов/переносов строки
all_items = []
for line in lines:
    items = [item.strip() for item in line.split(',') if item.strip() != '']
    all_items.extend(items)

# Создаем Series и считаем частоту
import pandas as pd
frequencies = pd.Series(all_items).value_counts()

# Выводим 59-е место (индекс 58)
print("Продукт на 59 месте:", frequencies.index[58])
print("Его частота:", frequencies.iloc[58])

Продукт на 59 месте: chewing gum
Его частота: 207


In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R
# Load the arules package
library(arules)

# Read the dataset into a transactions object
data <- read.transactions("groceries.csv", format="basket", sep=",")

# Mine association rules with specified minimum support and confidence
rules <- apriori(data, parameter = list(supp = 0.009, conf = 0.304, minlen = 2, maxlen = 3))

# Filter rules to find those with the exact or closest matching parameters
# Filtering for support close to 0.009
rules_filtered_support <- subset(rules, support > 0.0089 & support < 0.0091)

# Filtering for confidence exactly 0.304918, using a small range for robustness with floating points
rules_filtered_conf <- subset(rules_filtered_support, confidence > 0.304917 & confidence < 0.304919)

# Filtering for lift close to 2.797
rules_final <- subset(rules_filtered_conf, lift > 2.796 & lift < 2.798)

# Sort by lift for better presentation if multiple rules match
rules_final <- sort(rules_final, by = "lift", decreasing = TRUE)

# Display the LHS of the found rules
if (length(rules_final) > 0) {
  cat("Found itemset(s) (LHS) with specified parameters:\n")
  inspect(lhs(rules_final))
} else {
  cat("No itemset found matching the exact criteria.\n")
}

Apriori

Parameter specification:
 confidence minval smax arem  aval originalSupport maxtime support minlen
      0.304    0.1    1 none FALSE            TRUE       5   0.009      2
 maxlen target  ext
      3  rules TRUE

Algorithmic control:
 filter tree heap memopt load sort verbose
    0.1 TRUE TRUE  FALSE TRUE    2    TRUE

Absolute minimum support count: 88 

set item appearances ...[0 item(s)] done [0.00s].
set transactions ...[169 item(s), 9835 transaction(s)] done [0.01s].
sorting and recoding items ... [93 item(s)] done [0.00s].
creating transaction tree ... done [0.01s].
checking subsets of size 1 2 3 done [0.01s].
writing ... [164 rule(s)] done [0.00s].
creating S4 object  ... done [0.01s].
No itemset found matching the exact criteria.


In addition: Warning message:
In apriori(data, parameter = list(supp = 0.009, conf = 0.304, minlen = 2,  :
  Mining stopped (maxlen reached). Only patterns up to a length of 3 returned!


In [ ]:
%%R
library(arules)

# 1. Считываем данные
data <- read.transactions("groceries.csv", format="basket", sep=",")

# 2. Генерируем правила с запасом (снизим порог поддержки до 0.008)
rules <- apriori(data, parameter = list(supp = 0.008, conf = 0.30, minlen = 2))

# 3. Сортируем все найденные правила по лифту (от большего к меньшему)
rules_sorted <- sort(rules, by = "lift", decreasing = TRUE)

# 4. Выводим ТОП-20 правил, чтобы найти наше
inspect(head(rules_sorted, 20))

Apriori

Parameter specification:
 confidence minval smax arem  aval originalSupport maxtime support minlen
        0.3    0.1    1 none FALSE            TRUE       5   0.008      2
 maxlen target  ext
     10  rules TRUE

Algorithmic control:
 filter tree heap memopt load sort verbose
    0.1 TRUE TRUE  FALSE TRUE    2    TRUE

Absolute minimum support count: 78 

set item appearances ...[0 item(s)] done [0.00s].
set transactions ...[169 item(s), 9835 transaction(s)] done [0.01s].
sorting and recoding items ... [100 item(s)] done [0.00s].
creating transaction tree ... done [0.01s].
checking subsets of size 1 2 3 4 done [0.01s].
writing ... [205 rule(s)] done [0.00s].
creating S4 object  ... done [0.01s].
     lhs                         rhs                    support confidence   coverage     lift count
[1]  {beef,                                                                                         
      whole milk}             => {root vegetables}  0.008032537  0.3779904 0.021250

In [ ]:
%%R
# Фильтруем правила по точным критериям из вопроса №3
# Снижаем порог поддержки до 0.007, как рекомендует примечание в тесте
rules_q3 <- apriori(data, parameter = list(supp = 0.007, conf = 0.45, minlen = 2))

# Ищем правило, где лифт близок к 2.357
target_rule <- subset(rules_q3, round(lift, 3) == 2.357)

# Выводим результат
inspect(target_rule)

Apriori

Parameter specification:
 confidence minval smax arem  aval originalSupport maxtime support minlen
       0.45    0.1    1 none FALSE            TRUE       5   0.007      2
 maxlen target  ext
     10  rules TRUE

Algorithmic control:
 filter tree heap memopt load sort verbose
    0.1 TRUE TRUE  FALSE TRUE    2    TRUE

Absolute minimum support count: 68 

set item appearances ...[0 item(s)] done [0.00s].
set transactions ...[169 item(s), 9835 transaction(s)] done [0.01s].
sorting and recoding items ... [104 item(s)] done [0.00s].
creating transaction tree ... done [0.01s].
checking subsets of size 1 2 3 4 done [0.01s].
writing ... [81 rule(s)] done [0.00s].
creating S4 object  ... done [0.01s].
    lhs                        rhs                support     confidence
[1] {beef, root vegetables} => {other vegetables} 0.007930859 0.4561404 
    coverage   lift     count
[1] 0.01738688 2.357404 78   


In [ ]:
%%R
# 1. Генерируем правила с достаточно низкими порогами, чтобы поймать нужную пару
# Ставим поддержку пониже, так как sausage встречается не так часто
rules_q7 <- apriori(data, parameter = list(supp = 0.001, conf = 0.05, minlen = 2))

# 2. Фильтруем точечно: LHS должен быть "sausage", а RHS - "rolls/buns"
target_rule_q7 <- subset(rules_q7, lhs %in% "sausage" & rhs %in% "rolls/buns")

# 3. Выводим результат
inspect(target_rule_q7)

Apriori

Parameter specification:
 confidence minval smax arem  aval originalSupport maxtime support minlen
       0.05    0.1    1 none FALSE            TRUE       5   0.001      2
 maxlen target  ext
     10  rules TRUE

Algorithmic control:
 filter tree heap memopt load sort verbose
    0.1 TRUE TRUE  FALSE TRUE    2    TRUE

Absolute minimum support count: 9 

set item appearances ...[0 item(s)] done [0.00s].
set transactions ...[169 item(s), 9835 transaction(s)] done [0.02s].
sorting and recoding items ... [157 item(s)] done [0.00s].
creating transaction tree ... done [0.01s].
checking subsets of size 1 2 3 4 5 6 done [0.03s].
writing ... [37937 rule(s)] done [0.03s].
creating S4 object  ... done [0.01s].
      lhs                            rhs              support confidence    coverage      lift count
[1]   {sausage}                   => {rolls/buns} 0.030604982  0.3257576 0.093950178 1.7710480   301
[2]   {sausage,                                                               

In [ ]:
%%R
# Запускаем алгоритм с точными параметрами из вопроса
rules_q8 <- apriori(data, parameter = list(support = 0.03, confidence = 0.25, minlen = 2))

# Выводим общее количество созданных правил
print(length(rules_q8))

Apriori

Parameter specification:
 confidence minval smax arem  aval originalSupport maxtime support minlen
       0.25    0.1    1 none FALSE            TRUE       5    0.03      2
 maxlen target  ext
     10  rules TRUE

Algorithmic control:
 filter tree heap memopt load sort verbose
    0.1 TRUE TRUE  FALSE TRUE    2    TRUE

Absolute minimum support count: 295 

set item appearances ...[0 item(s)] done [0.00s].
set transactions ...[169 item(s), 9835 transaction(s)] done [0.01s].
sorting and recoding items ... [44 item(s)] done [0.00s].
creating transaction tree ... done [0.01s].
checking subsets of size 1 2 3 done [0.00s].
writing ... [15 rule(s)] done [0.00s].
creating S4 object  ... done [0.01s].
[1] 15
